# Paper 2: 8B generator arm on a T4

Run the cells in order, top to bottom.

**Before cell 1:** Runtime menu, Change runtime type, pick **T4 GPU**.
**You need:** `paper2-8b-handover.zip` (from Jarvis) ready to upload when a cell asks.
**Time:** roughly 1 to 1.5 hours including setup. Keep this tab open while the run cell works.
**If the session dies:** the run checkpoints every example to your Google Drive (cell 6). Reopen this notebook, re-run cells 1 to 6, then the run cell again. It resumes where it stopped.

## 1. Check the GPU (should say T4)

In [ ]:
!nvidia-smi

## 2. Upload the handover zip and unpack it

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()
zipname = [k for k in uploaded if k.endswith('.zip')][0]
os.makedirs('/content/paper2-8b', exist_ok=True)
with zipfile.ZipFile(zipname) as z:
    z.extractall('/content/paper2-8b')
print(sorted(os.listdir('/content/paper2-8b')))

## 3. Install packages (llama-cpp with CUDA support)

In [ ]:
!pip install -q scikit-learn joblib numpy
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 --force-reinstall --no-cache-dir
import llama_cpp
print('llama_cpp', llama_cpp.__version__)

**Only if cell 3 errored**, run this instead (builds from source, about 10 minutes):

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --force-reinstall --no-cache-dir

## 4. Download the model (4.9 GB, a few minutes on Colab)

In [ ]:
!mkdir -p /content/paper2-8b/models
!curl -L -o /content/paper2-8b/models/llama-3.1-8b-instruct-Q4_K_M.gguf https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
!ls -lh /content/paper2-8b/models/

## 5. Tell the runner to use the GPU

In [ ]:
!sed -i 's/n_threads=4,/n_threads=4, n_gpu_layers=35,/' /content/paper2-8b/run_8b_cached.py
!grep -n "n_gpu_layers" /content/paper2-8b/run_8b_cached.py

## 6. Checkpoint backup to your Drive (so a dead session loses nothing)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/paper2-8b', exist_ok=True)
link = '/content/paper2-8b/day3_8b_checkpoint.jsonl'
target = '/content/drive/MyDrive/paper2-8b/day3_8b_checkpoint.jsonl'
if not os.path.exists(link):
    if not os.path.exists(target):
        open(target, 'a').close()
    os.symlink(target, link)
print('checkpoint ->', os.path.realpath(link))

## 7. Run all 500 examples
Progress prints every 10 examples. On a T4 this should take under an hour. If it stops early, just run this cell again, it resumes from the checkpoint.

In [ ]:
%cd /content/paper2-8b
!python run_8b_cached.py --model-path models/llama-3.1-8b-instruct-Q4_K_M.gguf

## 8. Get the results
This shows the arm scores and downloads two files. **Send both files back to Jarvis** and the paper stats get finished from them.

In [ ]:
!cat /content/paper2-8b/day3_8b_results.json
from google.colab import files
files.download('/content/paper2-8b/day3_8b_results.json')
files.download('/content/paper2-8b/day3_8b_checkpoint.jsonl')